# Chapter 11: ODE workflow

Companion notebook. Read the written chapter at [chapter.html](chapter.html). Solutions at the bottom.


## Example 1: one Euler step by hand

In [ ]:
def f(t, y):
    return -2.0 * y

t0, y0, h = 0.0, 1.0, 0.1
y1 = y0 + h * f(t0, y0)
print(y1)
print(2.71828 ** (-2 * 0.1))


## Example 2: looped Euler with plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(t, y):
    return -2.0 * y

t0, tf, N = 0.0, 2.0, 20
h = (tf - t0) / N
t, y = t0, 1.0
ts, ys = [t], [y]
for _ in range(N):
    y = y + h * f(t, y)
    t = t + h
    ts.append(t); ys.append(y)

ts = np.array(ts); ys = np.array(ys)
plt.figure(figsize=(7, 4))
plt.plot(ts, ys, "o-", label="Euler, N=20")
plt.plot(ts, np.exp(-2 * ts), "k--", label="analytic")
plt.xlabel("t"); plt.ylabel("y"); plt.legend(); plt.show()


## Example 3: wrap as a function

In [ ]:
def euler(f, t0, y0, tf, N):
    h = (tf - t0) / N
    ts = [t0]; ys = [y0]
    t = t0; y = y0
    for _ in range(N):
        y = y + h * f(t, y)
        t = t + h
        ts.append(t); ys.append(y)
    return ts, ys

def f(t, y):
    return -2.0 * y

ts, ys = euler(f, 0, 1.0, 2.0, 20)
print(ys[-1])


## Example 4: class-based integrator

In [ ]:
import numpy as np
class ExplicitEuler:
    def __init__(self, f, t0, y0, tf, N):
        self.f = f; self.t0 = t0; self.y0 = y0
        self.tf = tf; self.N = N; self.h = (tf - t0) / N
    def solve(self):
        ts = np.zeros(self.N + 1); ys = np.zeros(self.N + 1)
        ts[0] = self.t0; ys[0] = self.y0
        for i in range(self.N):
            ys[i+1] = ys[i] + self.h * self.f(ts[i], ys[i])
            ts[i+1] = ts[i] + self.h
        return ts, ys

solver = ExplicitEuler(lambda t, y: -2 * y, 0.0, 1.0, 2.0, 100)
ts, ys = solver.solve()
print(ys[-1])


## Example 5: vector ODE, harmonic oscillator

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(t, y):
    x, v = y
    return np.array([v, -x])

def euler(f, t0, y0, tf, N):
    h = (tf - t0) / N
    ts = np.zeros(N + 1); ys = np.zeros((N + 1, len(y0)))
    ts[0] = t0; ys[0] = y0
    for i in range(N):
        ys[i+1] = ys[i] + h * f(ts[i], ys[i])
        ts[i+1] = ts[i] + h
    return ts, ys

ts, ys = euler(f, 0.0, np.array([1.0, 0.0]), 20.0, 500)
plt.figure(figsize=(7, 4))
plt.plot(ts, ys[:, 0], label="x(t)")
plt.plot(ts, np.cos(ts), "k--", label="analytic cos(t)")
plt.xlabel("t"); plt.ylabel("x"); plt.legend(); plt.show()


## Example 6: convergence study

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(t, y):
    return -2.0 * y

def euler_final(N):
    h = 2.0 / N
    y = 1.0; t = 0.0
    for _ in range(N):
        y = y + h * f(t, y); t = t + h
    return y

Ns = [10, 20, 40, 80, 160, 320, 640]
true = np.exp(-2 * 2.0)
errs = [abs(euler_final(N) - true) for N in Ns]
hs = [2.0 / N for N in Ns]
plt.figure(figsize=(6, 4))
plt.loglog(hs, errs, "o-", label="Euler error")
plt.loglog(hs, hs, "k--", label="slope 1 reference")
plt.xlabel("h"); plt.ylabel("|error at t=2|"); plt.legend()
plt.grid(True, which="both", alpha=0.3); plt.show()


## Example 7: stiffness blow-up

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

k = 100.0
def f(t, y):
    return -k * y

plt.figure(figsize=(7, 4))
for N in [50, 150, 500]:
    h = 1.0 / N
    y = 1.0; t = 0.0
    ts = [t]; ys = [y]
    for _ in range(N):
        y = y + h * f(t, y); t = t + h
        ts.append(t); ys.append(y)
    plt.plot(ts, ys, label=f"N={N}, h={h:.4f}")
plt.xlabel("t"); plt.ylabel("y"); plt.title("Stiff exponential decay, k = 100")
plt.legend(); plt.show()


## Example 8: skydiver

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

m, g, c = 70.0, 9.81, 0.25
def f(t, v):
    return g - (c / m) * v * v

t0, tf, N = 0.0, 30.0, 600
h = (tf - t0) / N
t, v = t0, 0.0
ts = [t]; vs = [v]
for _ in range(N):
    v = v + h * f(t, v); t = t + h
    ts.append(t); vs.append(v)
plt.figure(figsize=(7, 4))
plt.plot(ts, vs)
plt.axhline(np.sqrt(m * g / c), color="r", linestyle="--", label="terminal velocity")
plt.xlabel("t (s)"); plt.ylabel("v (m/s)"); plt.legend(); plt.show()


## Example 9: controlled pendulum (Lecture 16)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

b, K, Kp, target = 1.0, 9.81, 25.0, np.pi/4

def f(t, y):
    th, om = y
    return np.array([om, -b * om - K * np.sin(th) + Kp * (target - th)])

def euler_vec(f, t0, y0, tf, N):
    h = (tf - t0) / N
    ys = np.zeros((N+1, len(y0))); ts = np.zeros(N+1)
    ts[0] = t0; ys[0] = y0
    for i in range(N):
        ys[i+1] = ys[i] + h * f(ts[i], ys[i])
        ts[i+1] = ts[i] + h
    return ts, ys

ts, ys = euler_vec(f, 0.0, np.array([0.0, 0.0]), 10.0, 5000)
plt.figure(figsize=(7, 4))
plt.plot(ts, ys[:, 0] * 180/np.pi, label="theta (deg)")
plt.axhline(target * 180/np.pi, color="r", linestyle="--", label="target")
plt.xlabel("t (s)"); plt.ylabel("angle (deg)"); plt.legend(); plt.show()


## Example 10: Heun, second order

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def heun_final(N):
    h = 2.0 / N
    y = 1.0; t = 0.0
    f = lambda t, y: -2 * y
    for _ in range(N):
        k1 = f(t, y)
        k2 = f(t + h, y + h * k1)
        y = y + 0.5 * h * (k1 + k2)
        t = t + h
    return y

Ns = [10, 20, 40, 80, 160, 320]
true = np.exp(-2 * 2.0)
errs = [abs(heun_final(N) - true) for N in Ns]
hs = [2.0 / N for N in Ns]
plt.figure(figsize=(6, 4))
plt.loglog(hs, errs, "o-", label="Heun error")
plt.loglog(hs, [h**2 for h in hs], "k--", label="slope 2 reference")
plt.xlabel("h"); plt.ylabel("|error|"); plt.legend()
plt.grid(True, which="both", alpha=0.3); plt.show()


## Example 11: full class on a damped oscillator

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class ExplicitEuler:
    def __init__(self, f, t0, y0, tf, N):
        self.f = f; self.t0 = t0; self.y0 = np.asarray(y0, dtype=float)
        self.tf = tf; self.N = N; self.h = (tf - t0) / N
    def solve(self):
        dim = self.y0.shape
        ts = np.zeros(self.N + 1)
        ys = np.zeros((self.N + 1,) + dim)
        ts[0] = self.t0; ys[0] = self.y0
        for i in range(self.N):
            ys[i+1] = ys[i] + self.h * self.f(ts[i], ys[i])
            ts[i+1] = ts[i] + self.h
        return ts, ys

def damped(t, y):
    x, v = y
    return np.array([v, -0.2 * v - x])

solver = ExplicitEuler(damped, 0.0, [1.0, 0.0], 30.0, 3000)
ts, ys = solver.solve()
plt.figure(figsize=(7, 4))
plt.plot(ts, ys[:, 0])
plt.xlabel("t"); plt.ylabel("x"); plt.title("Damped oscillator"); plt.show()


## Example 12: energy drift check

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(t, y):
    x, v = y
    return np.array([v, -x])

def euler_vec(f, t0, y0, tf, N):
    h = (tf - t0) / N
    ys = np.zeros((N+1, len(y0))); ts = np.zeros(N+1)
    ts[0] = t0; ys[0] = y0
    for i in range(N):
        ys[i+1] = ys[i] + h * f(ts[i], ys[i])
        ts[i+1] = ts[i] + h
    return ts, ys

ts, ys = euler_vec(f, 0.0, np.array([1.0, 0.0]), 50.0, 5000)
energy = 0.5 * (ys[:, 0]**2 + ys[:, 1]**2)
plt.figure(figsize=(7, 4))
plt.plot(ts, energy)
plt.xlabel("t"); plt.ylabel("E")
plt.title("Energy under Euler integration (should be constant)")
plt.show()


---
## Solutions

### Problem 1: dy/dt = -y on [0, 5]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def euler(f, t0, y0, tf, N):
    h = (tf - t0) / N
    ts = np.zeros(N+1); ys = np.zeros(N+1)
    ts[0] = t0; ys[0] = y0
    for i in range(N):
        ys[i+1] = ys[i] + h * f(ts[i], ys[i])
        ts[i+1] = ts[i] + h
    return ts, ys

ts, ys = euler(lambda t, y: -y, 0.0, 1.0, 5.0, 50)
plt.figure(figsize=(7, 4))
plt.plot(ts, ys, label="Euler")
plt.plot(ts, np.exp(-ts), "k--", label="exp(-t)")
plt.legend(); plt.xlabel("t"); plt.ylabel("y"); plt.show()


### Problem 2: Heun (improved Euler) in a class

In [ ]:
import numpy as np
class Heun:
    def __init__(self, f, t0, y0, tf, N):
        self.f = f; self.t0 = t0; self.y0 = np.asarray(y0, dtype=float)
        self.tf = tf; self.N = N; self.h = (tf - t0) / N
    def solve(self):
        ts = np.zeros(self.N + 1); ys = np.zeros((self.N + 1,) + self.y0.shape)
        ts[0] = self.t0; ys[0] = self.y0
        for i in range(self.N):
            t, y, h = ts[i], ys[i], self.h
            k1 = self.f(t, y)
            k2 = self.f(t + h, y + h * k1)
            ys[i+1] = y + 0.5 * h * (k1 + k2)
            ts[i+1] = t + h
        return ts, ys

ts, ys = Heun(lambda t, y: -2 * y, 0.0, np.array(1.0), 2.0, 100).solve()
print(ys[-1])


### Problem 3: f signature bug

In [ ]:
# The function f was defined as f(y) but the loop calls f(t, y).
# Add the t parameter (even if unused).
def f(t, y):
    return y

t0, tf, N = 0, 1, 10
h = (tf - t0) / N
y = 1.0
for i in range(N):
    y = y + h * f(t0 + i*h, y)
print(y)  # close to e


### Problem 4: Lotka-Volterra

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

alpha, beta, gamma, delta = 1.0, 0.5, 1.0, 0.5

def f(t, y):
    x, z = y
    return np.array([alpha*x - beta*x*z, delta*x*z - gamma*z])

def euler(f, t0, y0, tf, N):
    h = (tf - t0) / N
    ts = np.zeros(N+1); ys = np.zeros((N+1, len(y0)))
    ts[0] = t0; ys[0] = y0
    for i in range(N):
        ys[i+1] = ys[i] + h * f(ts[i], ys[i])
        ts[i+1] = ts[i] + h
    return ts, ys

ts, ys = euler(f, 0.0, np.array([1.0, 1.0]), 30.0, 5000)
plt.figure(figsize=(8, 4))
plt.plot(ts, ys[:, 0], label="prey x")
plt.plot(ts, ys[:, 1], label="predator y")
plt.xlabel("t"); plt.legend(); plt.show()


### Problem 5: convergence study on the oscillator

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(t, y):
    x, v = y
    return np.array([v, -x])

def euler_vec_final(N):
    h = 10.0 / N
    y = np.array([1.0, 0.0])
    t = 0.0
    for _ in range(N):
        y = y + h * f(t, y); t = t + h
    return y

Ns = [100, 200, 400, 800, 1600]
true_x = np.cos(10.0)
errs = [abs(euler_vec_final(N)[0] - true_x) for N in Ns]
hs = [10.0 / N for N in Ns]
plt.figure(figsize=(6, 4))
plt.loglog(hs, errs, "o-", label="Euler error at t=10")
plt.loglog(hs, hs, "k--", label="slope 1")
plt.xlabel("h"); plt.ylabel("|error|"); plt.legend()
plt.grid(True, which="both", alpha=0.3); plt.show()
